In [1]:
# --- repo bootstrap: make `panelclv` importable and relative data paths resolve ---
import os, sys
from pathlib import Path
_root = Path.cwd().resolve()
while _root != _root.parent and not (_root / 'pyproject.toml').exists():
    _root = _root.parent
os.chdir(_root)                                 # so "Datasets/..." resolve from repo root
_src = _root / 'src'
if _src.exists() and str(_src) not in sys.path:  # fallback if panelclv isn't pip-installed
    sys.path.insert(0, str(_src))
# ---------------------------------------------------------------------------------

%reload_ext autoreload
%autoreload 2

In Terminal all the time
cd ~/Desktop/Thesis
source venvs/thesis_rocm/bin/activate

cd /home/virthian/Desktop/Thesis/Package/my_package/autoseqmodels
pip install -e .

In [2]:
# ---------------------------------------------------------------------------
# Standard library
# ---------------------------------------------------------------------------
import json
from pathlib import Path

# ---------------------------------------------------------------------------
# Third-party
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.model_selection import train_test_split

# ---------------------------------------------------------------------------
# This project (installed via `pip install -e .`, so no manual path setup).
# Models/__init__.py is the public API.
# ---------------------------------------------------------------------------
from panelclv.data_preparation import dynamic_panel_dataset
from panelclv.models import (
    MultinomialLSTMModel,  # softmax-over-counts LSTM (classifier head)
    compute_class_weights,  # class weights for weighted_ce / focal
    InferenceMultinomialLSTMModel,  # same backbone, sampling head
    mc_forecast,  # Valendin-style autoregressive holdout rollout
    mc_compute_metrics,  # RMSE / bias% / aggregate-MAPE
)
from panelclv.training import (
    fit_model,  # shared training loop (CE / weighted_ce / focal / emd)
)
from panelclv.tuning import (
    run_optuna_study,  # hyper-parameter + covariate search
)
from panelclv.evaluation import (
    weekly_actuals,
    weekly_aggregate_predictions,
    plot_weekly_aggregated,
    metrics_table,
    alignment_check,
    save_predictions_to_csv,
)
from panelclv.experiments import (
    make_data_builder,  # builds the Optuna data_builder closure
    build_inference_from_trial,  # rebuilds the winning model + loads its checkpoint
)

# Data Loading


### Panels configuration

In [7]:
from panelclv.configs.panel_config import PanelConfig

csv_path = "Datasets/Dataset_clean/electronics_customer_week_panel.csv"

# One config object replaces DATA_CONFIG + TIME_FEATURES + FEATURE_SCHEMA + INPUT_SPEC.
cfg = PanelConfig(
    # --- identity / target ---
    id_col="Id",
    target_col="Transactions",
    frequency="weekly",
    training_start="1999-01-01", training_end="2000-12-31",
    holdout_start="2001-01-01",  holdout_end="2001-12-31",
    time_cols=("year", "week"),
    clip_target_upper=6,

    # --- cohort selection (Valendin et al.) -----------------------------------
    # True  -> keep only customers with >=1 transaction during the calibration
    #          window (equivalently, first purchase <= training_end). Customers
    #          first seen only in the holdout are dropped. Applied inside
    #          prepare_dataset, so the LSTM and the Pareto/NBD benchmark score the
    #          SAME customers (fair comparison). Set False to keep every customer.
    require_calibration_activity=True,

    # --- engineered time features (OPT-IN: omit -> none are created) ---
    time_features={"add_year_idx": True, "add_week_sin_cos": True},
    #ar_features=("period_since_last_transaction", "active_in_last_3_periods"),

    # --- feature roles (the target is target_col; don't list it here) ---
    # week_sin / week_cos are auto-added to the 'time' role from the enabled
    # flags above; year_idx is placed explicitly (a trend, not cyclical).
    known_future=("year_idx", "high.season"),
    static=("Gender", "Income"),
    observed_past=(),  # no lags or rolling stats in this example, but could add here

    # --- which columns to embed; "auto" infers cardinality from calibration ---
    #   Transactions -> clip_target_upper + 1 = 7
    #   Gender / Income -> (calibration max + 1)   (pin an int to fix the size)
    embedded_cols={"Transactions": "auto", "Gender": "auto", "Income": "auto"},
)


In [4]:
from panelclv.configs.panel_config import PanelConfig

csv_path = "Datasets/Dataset_clean/electronicV2_customer_month_panel.csv"

# One config object replaces DATA_CONFIG + TIME_FEATURES + FEATURE_SCHEMA + INPUT_SPEC.
cfg = PanelConfig(
    # --- identity / target ---
    id_col="Id",
    target_col="Transactions",
    frequency="monthly",
    training_start="1999-01-01", training_end="2000-12-31",
    holdout_start="2001-01-01",  holdout_end="2001-12-31",
    time_cols=("year", "month"),
    clip_target_upper=6,

    # --- cohort selection (Valendin et al.) -----------------------------------
    # True  -> keep only customers with >=1 transaction during the calibration
    #          window (equivalently, first purchase <= training_end). Customers
    #          first seen only in the holdout are dropped. Applied inside
    #          prepare_dataset, so the LSTM and the Pareto/NBD benchmark score the
    #          SAME customers (fair comparison). Set False to keep every customer.
    require_calibration_activity=True,

    # --- engineered time features (OPT-IN: omit -> none are created) ---
    time_features={"add_year_idx": True, "add_month_sin_cos": True},
    #ar_features=("period_since_last_transaction", "active_in_last_3_periods"),

    # --- feature roles (the target is target_col; don't list it here) ---
    # month_sin / month_cos are auto-added to the 'time' role from the enabled
    # flags above; year_idx is placed explicitly (a trend, not cyclical).
    known_future=("year_idx", "high.season"),
    static=("Gender", "Income"),
    observed_past=(),  # no lags or rolling stats in this example, but could add here

    # --- which columns to embed; "auto" infers cardinality from calibration ---
    #   Transactions -> clip_target_upper + 1 = 7
    #   Gender / Income -> (calibration max + 1)   (pin an int to fix the size)
    embedded_cols={"Transactions": "auto", "Gender": "auto", "Income": "auto"},
)


In [5]:
# ---- Monte Carlo forecast settings ----------------------------------------
# Single source of truth for the holdout simulation count: every mc_forecast
# call below uses N_SIMULATIONS, so all models are compared on equal footing.
# Higher -> smoother per-customer mean, more compute. MC_SEED makes the
# forecast reproducible (same model + data + seed -> identical sampled paths).
N_SIMULATIONS = 600
MC_SEED = 42

In [ ]:
# ---- 1. Build the (N, T, F) arrays from your config ----------------------
panel     = pd.read_csv(csv_path)
data_full = dynamic_panel_dataset.prepare_dataset(panel, cfg)

# ---- 2. Shape tensors for fit_model --------------------------------------
# samples : (N, T-1, F) float32
# targets : fit_model wants (B, T) long with values in [0, max_trans)
X = data_full["samples"]
y = data_full["targets"].squeeze(-1).astype(np.int64)

# max_trans comes from the RESOLVED spec prepare_dataset returns (handles 'auto').
max_trans = data_full["input_spec"]["embedded_cols"][data_full["target_col"]]
assert y.min() >= 0 and y.max() < max_trans, (
    f"Transactions in [{y.min()}, {y.max()}] but the target embedding caps at {max_trans-1}. "
    f"Raise the cardinality or clip the panel."
)

# ---- 3. Customer-wise train/val split -----------------------------------
train_idx, val_idx = train_test_split(
    np.arange(data_full["N"]), test_size=0.1, random_state=42,
)


N=829 T_CAL=104 T_HOLD=52 F=7
seq_cols   = ['Transactions', 'week_sin', 'week_cos', 'year_idx', 'high.season', 'Gender', 'Income']
target_col = 'Transactions' at index 0
calibration (829, 104, 7) | samples (829, 103, 7) | targets (829, 103, 1) | holdout (829, 52, 7)
embedded_cols = {'Transactions': 7, 'Gender': 2, 'Income': 10}


## LSTM training & hyperparameter search

### LSTM with Optuna

### Loss trials

In [9]:
# ---- 5. (Optional) class weights for focal / weighted_ce -----------------
# Skip the next two lines for the paper-faithful CE run.
LOSS_TYPE     = "cross_entropy"            # 'cross_entropy' | 'weighted_ce' | 'focal' | 'emd'
class_weights = compute_class_weights(y[train_idx], num_classes=max_trans)
print("class_weights:", class_weights.tolist())

class_weights: [0.0023855797480791807, 0.2213481217622757, 0.3406653106212616, 0.7514676451683044, 1.5286264419555664, 2.484017848968506, 1.671488642692566]


In [10]:
# ---- 6. data_builder for Optuna -----------------------------------------
# `make_data_builder` returns the closure run_optuna_study calls each trial: it
# slices data_full to the trial's feature subset and builds train/val DataLoaders
# on the SAME customer split (train_idx / val_idx). Replaces the hand-written
# closure -- identical behaviour, one line.
data_builder = make_data_builder(data_full, train_idx, val_idx)

## Optuna optimization cross entropy

In [12]:
# ---- 7. Run the LSTM Optuna study ---------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
STUDY_NAME = f"lstm_{LOSS_TYPE}"            # one study per loss → no schema collisions

lstm_study = run_optuna_study(
    model_type="lstm",
    data_builder=data_builder,
    data_info={
        "n_epochs":       150,
        "patience":       7,
        "checkpoint_dir": "./checkpoints/lstm_optuna",
        "verbose":        False,
        # Loss configuration (all four read from data_info — unused keys are ignored).
        "loss_type":      LOSS_TYPE,
        "class_weights":  class_weights    # used by 'weighted_ce' / 'focal'; harmless for 'cross_entropy'
    },
    removable_features=["Gender", "Income", "high.season", "year_idx", ("week_sin", "week_cos")],  # Optuna can drop these if it wants; the LSTM will still get the seq/time features
    #removable_features=["Gender", "Income", "high.season", "year_idx", ("month_sin", "month_cos")],  # Optuna can drop these if it wants; the LSTM will still get the seq/time features
    device=device,
    n_trials=200,                            # 64 archs × 9 dropout points × 3 batches — give TPE room
    study_name=STUDY_NAME,
    summary_dir="./optuna_summaries",
    storage="sqlite:///optuna_summaries/aug_lstm_pareto.db", #new line
)

print("best trial   :", lstm_study.best_trial.number)
print("best val loss:", lstm_study.best_trial.value)
print("best params  :", lstm_study.best_trial.params)
print("checkpoint   :", lstm_study.best_trial.user_attrs["checkpoint_path"])

[I 2026-05-31 21:54:27,205] A new study created in RDB with name: lstm_cross_entropy_20260531_2154
[W 2026-05-31 21:55:00,717] Trial 0 failed with parameters: {'hidden_dim': 128, 'memory_units': 32, 'dense_units': 64, 'dropout': 0.2832290311184182, 'learning_rate': 0.00010725209743172001, 'weight_decay': 0.00757947995334801, 'batch_size': 64, 'use_Gender': False, 'use_Income': True, 'use_high.season': False, 'use_year_idx': False, 'use_week_sin+week_cos': False} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/virthian/Desktop/Thesis/venvs/thesis_rocm/lib/python3.12/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/home/virthian/Desktop/Thesis/Package_Notebook_refactored/Models/optuna_tuning.py", line 762, in <lambda>
    lambda trial: objective(
                  ^^^^^^^^^^
  File "/home/virthian/Desktop/Thesis/Package_Notebook_refactored/Mo

KeyboardInterrupt: 

### Optuna with rollout

In [ ]:
# ---- 7. Run the LSTM Optuna study ---------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
STUDY_NAME = f"lstm_{LOSS_TYPE}_rollout_composite"            # one study per loss → no schema collisions

lstm_study = run_optuna_study(
    model_type="lstm",
    selection_metric="rollout_composite",  # new line: optimize the metric we actually care about, not just val loss
    rollout_data=data_full,                        # new line: pass the full data for the rollout metric; the study will split it internally
    val_idx=val_idx,                                # new line: pass val_idx so the study can compute the rollout metric on the same customers as the val loss
    rollout_horizon=52,
    rollout_n_simulations=100,
    data_builder=data_builder,
    data_info={
        "n_epochs":       150,
        "patience":       7,
        "checkpoint_dir": "./checkpoints/lstm_optuna",
        "verbose":        False,
        # Loss configuration (all four read from data_info — unused keys are ignored).
        "loss_type":      LOSS_TYPE,
        "class_weights":  class_weights    # used by 'weighted_ce' / 'focal'; harmless for 'cross_entropy'
    },
    removable_features=["Gender", "Income", "high.season", "year_idx", ("week_sin", "week_cos")],  # Optuna can drop these if it wants; the LSTM will still get the seq/time features
    device=device,
    n_trials=200,                            # 64 archs × 9 dropout points × 3 batches — give TPE room
    study_name=STUDY_NAME,
    summary_dir="./optuna_summaries",
    storage="sqlite:///optuna_summaries/aug_lstm_pareto.db", #new line
)

print("best trial   :", lstm_study.best_trial.number)
print("best val loss:", lstm_study.best_trial.value)
print("best params  :", lstm_study.best_trial.params)
print("checkpoint   :", lstm_study.best_trial.user_attrs["checkpoint_path"])

In [ ]:
# ---- 8. Rebuild the LSTM with the Optuna-selected arch + load weights ----
# `build_inference_from_trial` reads the best trial's architecture + checkpoint,
# slices data_full to that trial's feature subset, rebuilds the matching inference
# model (mode="sample"), and loads the weights. It returns BOTH the model and the
# sliced `data_best` -- feed data_best (not data_full) to the forecaster so the
# feature columns match the trained weights.
inference_model, data_best = build_inference_from_trial(lstm_study, data_full, "lstm")

In [ ]:
# ---- 9. Valendin-style autoregressive MC forecast ------------------------
forecast = mc_forecast(
    inference_model,
    data_best,
    n_simulations=N_SIMULATIONS,
    device=device,
    seed=MC_SEED,
)

# Guard against a stale (not-re-run) cell silently reporting an old count.
assert forecast["simulations"].shape[0] == forecast["n_simulations"] == N_SIMULATIONS

print("simulations shape:", forecast["simulations"].shape)        # (S, N, T_HOLD)
print("prediction mean  :", forecast["prediction_mean"].shape)    # (N, T_HOLD)
print("actual (real)    :", forecast["actual"].shape)             # (N, T_HOLD)

In [ ]:
# ---- 10. Score + sanity check -------------------------------------------
metrics = mc_compute_metrics(forecast["actual"], forecast["prediction_mean"])
print(metrics)

# Aggregate predicted vs actual per week (useful for the thesis plot).
agg_pred   = forecast["prediction_mean"].sum(axis=0)              # (T_HOLD,)
agg_actual = forecast["actual"].sum(axis=0)
for i in range(min(20, len(agg_pred))):
    print(f"  week {i:>2}  pred={agg_pred[i]:6.1f}  actual={agg_actual[i]:6.1f}")

In [ ]:
# Aggregate actuals across customers per holdout week.
actuals = forecast["actual"].sum(axis=0)            # (T_HOLD,)

# (Optional) training-window aggregate to show context to the left of the holdout.
# data_full["calibration"] is (N, T_CAL, F); the target column lives at target_idx.
target_idx    = data_full["seq_cols"].index(data_full["target_col"])
train_actuals = data_full["calibration"][..., target_idx].sum(axis=0)   # (T_CAL,)

# Add a trailing 1-axis so weekly_aggregate_predictions takes the (S, N, T, 1)
# branch and draws the 95% MC ribbon.
mc_sims = forecast["simulations"][..., None]        # (S, N, T_HOLD, 1)

fig, ax = plot_weekly_aggregated(
				actuals=actuals,
    data=data_best,
    pareto_paper_benchmark=True,      
				predictions_by_model={"LSTM (MC mean + 95% CI)": mc_sims},
				train_actuals=train_actuals,                    # omit to plot only the holdout
				title="Bank panel — weekly aggregate transactions",
				show_ci=True,
				# save_path="figures/bank_lstm_weekly.png",    # uncomment to save
)


In [ ]:
tbl = metrics_table(
				forecast["actual"],
				{"LSTM": forecast["simulations"][..., None]},
				pareto_nbd_benchmark=True,
				pareto_paper_benchmark=True,
				data=data_best,
)
print(tbl)

In [ ]:
import numpy as np
from panelclv.evaluation.plot_utils import _pareto_from_data
A   = forecast["actual"].sum()                     # total actual holdout tx
L   = forecast["simulations"].mean(0).sum()        # LSTM total
P   = _pareto_from_data(data_best, "mle").sum()    # Pareto MLE total
H   = _pareto_from_data(data_best, "paper").sum()  # Pareto HB total
print(f"actual={A:.0f}  LSTM={L:.0f}  Pareto={P:.0f}  HB={H:.0f}")
print(f"calib total/wk={data_best['calibration'][...,0].sum()/data_best['calibration'].shape[1]:.0f}  "
						f"holdout total/wk={A/forecast['actual'].shape[1]:.0f}")
